# P-Task6: Server Log Error Streamer

**Problem:** backend team ki server log files itni badi hain ke memory mein load nahi ho sakti. Mujhe aisa
streaming system banana hai jo file line by line padhe, sirf critical error entries yield kare, unhein error type
ke hisaab se group kare aur live running count dikhaye, **poori file ek saath load kiye bina**.

**Concepts used:** iterator protocol (`__iter__`, `__next__`, `StopIteration`), generator functions (`yield`),
generator expressions, `itertools` (`chain`, `islice`, `cycle`, `product`), memory efficiency.

## 1. Iteration andar se kaise chalti hai
`for` loop asal mein `iter()` aur `next()` call karta hai, jab `StopIteration` aata hai to ruk jata hai.

In [1]:
numbers = [10, 20, 30]
it = iter(numbers)
print(next(it))
print(next(it))
print(next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration aa gaya, iterator khatam")

10
20
30
StopIteration aa gaya, iterator khatam


## 2. Apna iterator class
`__iter__` khud object return karta hai, `__next__` agla value deta hai aur khatam hone par `StopIteration` raise karta hai.

In [2]:
class CountDown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


print(list(CountDown(5)))

[5, 4, 3, 2, 1]


## 3. Badi log file banana
Asli server log nahi thi, isliye 100,000 lines ki fake log file khud bana rahi hoon (random, magar `seed` fix
kiya taake har baar same result aaye).

In [3]:
import os
import random
import sys
from collections import Counter
from itertools import chain, cycle, islice, product

random.seed(42)
log_file = "server.log"

levels = ["INFO"] * 80 + ["DEBUG"] * 10 + ["WARNING"] * 5 + ["ERROR"] * 4 + ["CRITICAL"] * 1
error_types = ["DatabaseTimeout", "NullPointer", "DiskFull", "AuthFailure", "ConnectionReset", "OutOfMemory"]
total_lines = 100_000

with open(log_file, "w") as f:
    for i in range(total_lines):
        level = random.choice(levels)
        err = random.choice(error_types) if level in ("ERROR", "CRITICAL") else "-"
        f.write(f"2026-10-20 10:{i // 1700 % 60:02d}:{i % 60:02d} | {level} | {err} | request {i} processed\n")

print("File size:", round(os.path.getsize(log_file) / 1024 / 1024, 2), "MB")

File size: 5.51 MB


## 4. Generator functions
`yield` function ko pause kar deta hai, isliye ek waqt mein sirf ek line memory mein hoti hai.

In [4]:
def read_lines(path):
    with open(path) as f:
        for line in f:                 # file object khud lazy hai, ek ek line deta hai
            yield line.rstrip("\n")


def critical_entries(lines):
    """Sirf ERROR / CRITICAL lines yield karta hai, dict bana kar."""
    for line in lines:
        time, level, err, message = [part.strip() for part in line.split("|")]
        if level in ("ERROR", "CRITICAL"):
            yield {"time": time, "level": level, "type": err, "message": message}


first_three = list(islice(critical_entries(read_lines(log_file)), 3))
for entry in first_three:
    print(entry)

{'time': '2026-10-20 10:00:36', 'level': 'ERROR', 'type': 'NullPointer', 'message': 'request 36 processed'}
{'time': '2026-10-20 10:00:43', 'level': 'ERROR', 'type': 'DiskFull', 'message': 'request 43 processed'}
{'time': '2026-10-20 10:00:11', 'level': 'ERROR', 'type': 'DiskFull', 'message': 'request 71 processed'}


## 5. Live running count
Pipeline: `read_lines -> critical_entries -> counter`. Har entry ke saath count update hota hai aur har 500 entries
par screen par live count print hota hai.

In [5]:
counts = Counter()
level_counts = Counter()
seen = 0

for entry in critical_entries(read_lines(log_file)):
    seen += 1
    counts[entry["type"]] += 1
    level_counts[entry["level"]] += 1
    if seen % 500 == 0:
        print(f"[live] {seen:>5} errors so far | top: {counts.most_common(1)[0]}")

print("\nFinal count by error type:")
for err, n in counts.most_common():
    print(f"  {err:<16}{n:>5}  {'#' * (n // 60)}")
print("By level:", dict(level_counts))

[live]   500 errors so far | top: ('ConnectionReset', 94)
[live]  1000 errors so far | top: ('ConnectionReset', 176)
[live]  1500 errors so far | top: ('ConnectionReset', 268)
[live]  2000 errors so far | top: ('ConnectionReset', 346)
[live]  2500 errors so far | top: ('AuthFailure', 435)
[live]  3000 errors so far | top: ('AuthFailure', 521)
[live]  3500 errors so far | top: ('AuthFailure', 614)
[live]  4000 errors so far | top: ('AuthFailure', 698)
[live]  4500 errors so far | top: ('AuthFailure', 780)
[live]  5000 errors so far | top: ('AuthFailure', 874)

Final count by error type:
  AuthFailure       880  ##############
  ConnectionReset   846  ##############
  DiskFull          843  ##############
  NullPointer       830  #############
  OutOfMemory       822  #############
  DatabaseTimeout   814  #############
By level: {'ERROR': 4074, 'CRITICAL': 961}


## 6. Memory ka farq: list vs generator
`sys.getsizeof` se dekha ke list poori lines store karti hai jab ke generator sirf apna chhota sa object.

In [6]:
as_list = [line for line in read_lines(log_file)]
as_generator = (line for line in read_lines(log_file))      # generator expression

print("List ki size     :", round(sys.getsizeof(as_list) / 1024, 1), "KB (sirf pointers, lines alag)")
print("Generator ki size:", sys.getsizeof(as_generator), "bytes")
del as_list

List ki size     : 782.2 KB (sirf pointers, lines alag)
Generator ki size: 192 bytes


## 7. `itertools` ke chhote examples

In [7]:
# chain: do generators ko jod do
print(list(chain(islice(read_lines(log_file), 2), ["--- beech ---"], islice(read_lines(log_file), 1))))

# islice: sirf pehli kuch entries
print([e["type"] for e in islice(critical_entries(read_lines(log_file)), 5)])

# cycle: servers ko round robin assign karna
servers = cycle(["server-A", "server-B", "server-C"])
print([next(servers) for _ in range(7)])

# product: sab level aur type ke combos
print(len(list(product(["ERROR", "CRITICAL"], error_types))), "combinations")

['2026-10-20 10:00:00 | DEBUG | - | request 0 processed', '2026-10-20 10:00:01 | INFO | - | request 1 processed', '--- beech ---', '2026-10-20 10:00:00 | DEBUG | - | request 0 processed']
['NullPointer', 'DiskFull', 'DiskFull', 'DatabaseTimeout', 'ConnectionReset']
['server-A', 'server-B', 'server-C', 'server-A', 'server-B', 'server-C', 'server-A']
12 combinations


## 8. Safai
Badi log file repo mein nahi rakhni, isliye delete kar di.

In [8]:
os.remove(log_file)
os.path.exists(log_file)

False

## Conclusion
100,000 lines ki file ko line by line stream kiya aur sirf error entries nikaali. Count live update hota raha
aur kabhi poori file memory mein load nahi hui. Is task se maine seekha ke generator pipeline kaise banti hai
aur `yield` memory kaise bachata hai.